In [1]:
"""
Rotten Tomatoes 상세 스크래퍼
==============================
수집 데이터:
  [브라우즈 페이지]
  - 제목, 토마토미터(%), 관객점수/Popcornmeter(%), 스트리밍 날짜
  - Certified Fresh 여부, Verified Hot 여부 (필터 페이지이므로 모두 True)

  [개별 영화 페이지]
  - 장르(Genre), 관람등급(Rating: G/PG/PG-13/R 등)
  - 상영 시간(Runtime), 개봉일(Release Date)
  - 감독(Director), 주요 출연진(Cast)
  - 시놉시스(Synopsis)
  - 비평가 리뷰 수(Critic Reviews Count)
  - 관객 평점 수(Audience Ratings Count)
  - 배급사(Studio/Distributor)

설치:
    pip install requests beautifulsoup4 pandas
"""

import re
import json
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from concurrent.futures import ThreadPoolExecutor, as_completed

# ══════════════════════════════════════════════════════════
# 설정
# ══════════════════════════════════════════════════════════

BASE_URL = "https://www.rottentomatoes.com"
BROWSE_URL = (
    BASE_URL
    + "/browse/movies_at_home/audience:verified_hot~critics:certified_fresh"
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.google.com/",
}

# 개별 페이지 병렬 요청 스레드 수 (너무 높이면 차단 위험)
MAX_WORKERS = 3
# 요청 간 랜덤 딜레이 (초) — 차단 방지
DELAY_MIN = 1.0
DELAY_MAX = 2.5


# ══════════════════════════════════════════════════════════
# 공통 유틸
# ══════════════════════════════════════════════════════════

def fetch_soup(url: str, retries: int = 3) -> BeautifulSoup | None:
    """URL을 요청하고 BeautifulSoup을 반환합니다. 실패 시 재시도."""
    for attempt in range(1, retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "html.parser")
        except requests.RequestException as e:
            print(f"  [요청 오류] 시도 {attempt}/{retries}: {e}")
            if attempt < retries:
                time.sleep(2 * attempt)
    return None


def clean(text: str) -> str:
    """불필요한 공백·개행 제거"""
    return re.sub(r"\s+", " ", text or "").strip()


# ══════════════════════════════════════════════════════════
# STEP 1: 브라우즈 페이지 → 영화 기본 목록 수집
# ══════════════════════════════════════════════════════════

def parse_browse_page(soup: BeautifulSoup) -> list[dict]:
    """
    브라우즈 페이지의 <a href="/m/..."> 카드에서 기본 정보 추출.
    ▸ 제목, 토마토미터, Popcornmeter, 스트리밍 날짜, RT 링크
    ▸ 이 URL은 Certified Fresh + Verified Hot 필터 결과이므로 두 값 모두 True
    """
    movies = []
    seen = set()

    for link in soup.find_all("a", href=re.compile(r"^/m/")):
        href = link.get("href", "")
        lines = [
            l.strip()
            for l in link.get_text(separator="\n").splitlines()
            if l.strip()
        ]
        if not lines:
            continue

        title        = ""
        tomatometer  = ""
        popcornmeter = ""
        stream_date  = ""

        for line in lines:
            if re.match(r"^\d{1,3}%$", line):
                if not tomatometer:
                    tomatometer = line
                elif not popcornmeter:
                    popcornmeter = line
            elif re.match(r"^(Streaming|In Theaters|Opens?|DVD|Digital)", line, re.I):
                stream_date = line
            elif not re.match(r"^[\d%\W]+$", line) and len(line) > 2:
                if not title:
                    title = line

        if title and title not in seen and (tomatometer or popcornmeter):
            seen.add(title)
            movies.append({
                "제목":              title,
                "토마토미터(%)":      tomatometer,
                "Certified Fresh":  "✅" if tomatometer else "",
                "Popcornmeter(%)":  popcornmeter,
                "Verified Hot":     "✅" if popcornmeter else "",
                "스트리밍 날짜":     stream_date,
                "RT 링크":          BASE_URL + href,
                "_path":            href,   # 내부 용도 (저장 제외)
            })

    return movies


# ══════════════════════════════════════════════════════════
# STEP 2: 개별 영화 페이지 → 상세 정보 수집
# ══════════════════════════════════════════════════════════

def parse_movie_detail(path: str) -> dict:
    """
    /m/<movie_id> 페이지에서 상세 데이터 추출.
    ▸ 우선순위: ld+json 구조화 데이터 → HTML 파싱 보완
    """
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
    url  = BASE_URL + path
    soup = fetch_soup(url)

    detail = {
        "장르(Genre)":           "",
        "관람등급(Rating)":       "",
        "상영 시간(Runtime)":     "",
        "개봉일(Release Date)":   "",
        "감독(Director)":         "",
        "주요 출연진(Cast)":       "",
        "배급사(Studio)":          "",
        "시놉시스(Synopsis)":      "",
        "비평가 리뷰 수":           "",
        "관객 평점 수":             "",
    }

    if soup is None:
        return detail

    # ── [A] application/ld+json 구조화 데이터 파싱 (가장 안정적)
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
            if not isinstance(data, dict):
                continue
            if data.get("@type") not in ("Movie", "VideoObject"):
                continue

            # 장르
            genres = data.get("genre", [])
            detail["장르(Genre)"] = ", ".join(genres) if isinstance(genres, list) else genres

            # 관람등급
            detail["관람등급(Rating)"] = data.get("contentRating", "")

            # 상영 시간 (ISO 8601 → 가독성)
            duration = data.get("duration", "")
            if duration:
                match = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?", duration)
                if match:
                    h, m = match.group(1), match.group(2)
                    detail["상영 시간(Runtime)"] = (
                        f"{h}시간 {m}분" if h and m
                        else f"{h}시간" if h
                        else f"{m}분"
                    )

            # 개봉일
            detail["개봉일(Release Date)"] = data.get("datePublished", "")

            # 감독
            director = data.get("director", {})
            if isinstance(director, list):
                detail["감독(Director)"] = ", ".join(d.get("name", "") for d in director)
            elif isinstance(director, dict):
                detail["감독(Director)"] = director.get("name", "")

            # 출연진 (최대 5명)
            actors = data.get("actor", [])
            if isinstance(actors, list):
                detail["주요 출연진(Cast)"] = ", ".join(
                    a.get("name", "") for a in actors[:5]
                )

            # 시놉시스
            detail["시놉시스(Synopsis)"] = clean(data.get("description", ""))

            break  # 첫 번째 Movie 타입 ld+json 사용
        except Exception:
            continue

    # ── [B] score-board 웹 컴포넌트 (리뷰 수, 미수집 항목 보완)
    sb = soup.find("score-board") or soup.find("score-board-deprecated")
    if sb:
        if not detail["비평가 리뷰 수"]:
            detail["비평가 리뷰 수"] = sb.get("tomatometer-count", "")
        if not detail["관객 평점 수"]:
            detail["관객 평점 수"]   = sb.get("audience-count", "")
        if not detail["관람등급(Rating)"]:
            detail["관람등급(Rating)"] = sb.get("rating", "")

    # ── [C] data-qa 속성 기반 보완
    qa_map = {
        "비평가 리뷰 수": ["tomatometer-count", "critic-count"],
        "관객 평점 수":   ["audience-count", "popcornmeter-count"],
    }
    for field, qa_keys in qa_map.items():
        if not detail[field]:
            for key in qa_keys:
                el = soup.find(attrs={"data-qa": key})
                if el:
                    detail[field] = clean(el.get_text())
                    break

    # ── [D] 텍스트 레이블 기반 보완 (메타 정보 섹션)
    def find_by_label(pattern: str) -> str:
        """레이블 패턴과 매칭되는 요소 옆의 값을 추출"""
        for el in soup.find_all(string=re.compile(pattern, re.I)):
            parent = el.find_parent()
            if not parent:
                continue
            # 형제 태그
            for sib in parent.find_next_siblings():
                txt = clean(sib.get_text())
                if txt and txt.lower() != clean(el).lower():
                    return txt
        return ""

    if not detail["장르(Genre)"]:
        detail["장르(Genre)"]       = find_by_label(r"^genres?$")
    if not detail["관람등급(Rating)"]:
        detail["관람등급(Rating)"]  = find_by_label(r"^rating$")
    if not detail["상영 시간(Runtime)"]:
        detail["상영 시간(Runtime)"] = find_by_label(r"^(runtime|movie length)$")
    if not detail["개봉일(Release Date)"]:
        detail["개봉일(Release Date)"] = find_by_label(r"^(release\s*date|theater\s*date)$")
    if not detail["배급사(Studio)"]:
        detail["배급사(Studio)"]    = find_by_label(r"^(distributor|studio)$")
    if not detail["감독(Director)"]:
        detail["감독(Director)"]    = find_by_label(r"^(directed?\s*by|director)$")

    return detail


# ══════════════════════════════════════════════════════════
# STEP 3: 병렬 상세 스크래핑 & merge
# ══════════════════════════════════════════════════════════

def enrich_movies(movies: list[dict]) -> list[dict]:
    """
    ThreadPoolExecutor로 개별 영화 페이지를 병렬 스크래핑 후
    기본 데이터에 merge합니다.
    """
    total    = len(movies)
    enriched = [None] * total

    est_min = total * DELAY_MIN // MAX_WORKERS
    est_max = total * DELAY_MAX // MAX_WORKERS
    print(f"\n📄 개별 상세 페이지 스크래핑 ({total}편)")
    print(f"   예상 소요 시간: {int(est_min)}~{int(est_max)}초\n")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_map = {
            executor.submit(parse_movie_detail, m["_path"]): i
            for i, m in enumerate(movies)
        }

        for future in as_completed(future_map):
            idx  = future_map[future]
            base = {k: v for k, v in movies[idx].items() if k != "_path"}
            try:
                base.update(future.result())
            except Exception as e:
                print(f"  [오류] {movies[idx]['제목']}: {e}")
            enriched[idx] = base
            done = sum(1 for x in enriched if x is not None)
            print(f"  [{done:>3}/{total}] ✓ {base['제목']}")

    return [m for m in enriched if m is not None]


# ══════════════════════════════════════════════════════════
# STEP 4: 저장
# ══════════════════════════════════════════════════════════

COLUMN_ORDER = [
    "제목",
    "토마토미터(%)",   "Certified Fresh",
    "Popcornmeter(%)", "Verified Hot",
    "장르(Genre)",     "관람등급(Rating)",
    "상영 시간(Runtime)", "개봉일(Release Date)", "스트리밍 날짜",
    "감독(Director)",  "주요 출연진(Cast)",
    "배급사(Studio)",
    "시놉시스(Synopsis)",
    "비평가 리뷰 수",  "관객 평점 수",
    "RT 링크",
]


def save(movies: list[dict], filename: str = "rt_certified_fresh_full.csv"):
    if not movies:
        print("⚠ 저장할 데이터가 없습니다.")
        return

    df = pd.DataFrame(movies)
    cols = [c for c in COLUMN_ORDER if c in df.columns]
    df   = df[cols]

    # 토마토미터 내림차순 정렬
    df["_sort"] = pd.to_numeric(
        df["토마토미터(%)"].str.replace("%", ""), errors="coerce"
    )
    df = df.sort_values("_sort", ascending=False).drop(columns=["_sort"])
    df = df.reset_index(drop=True)

    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\n✅  저장 완료 → '{filename}'  ({len(df)}편)")

    # 미리보기
    preview = ["제목", "토마토미터(%)", "Popcornmeter(%)",
               "장르(Genre)", "관람등급(Rating)", "상영 시간(Runtime)", "감독(Director)"]
    preview = [c for c in preview if c in df.columns]
    print("\n[ 미리보기 (상위 10개) ]")
    print(df[preview].head(10).to_string(index=False))
    return df


# ══════════════════════════════════════════════════════════
# 메인
# ══════════════════════════════════════════════════════════

def main():
    print("=" * 62)
    print("🍅  Rotten Tomatoes 상세 스크래퍼")
    print("    필터: Certified Fresh (비평가) + Verified Hot (관객)")
    print("=" * 62)

    # STEP 1
    print("\n[1/3] 브라우즈 페이지 로드 중...")
    soup = fetch_soup(BROWSE_URL)
    if not soup:
        print("❌ 페이지 요청 실패.")
        return

    movies = parse_browse_page(soup)
    if not movies:
        print("❌ 영화 목록 파싱 실패.")
        return

    print(f"✅ {len(movies)}편 발견")
    for m in movies[:5]:
        print(f"   • {m['제목']}  🍅{m['토마토미터(%)']}  🍿{m['Popcornmeter(%)']}")
    if len(movies) > 5:
        print(f"   ... 외 {len(movies) - 5}편")

    # STEP 2 & 3
    print("\n[2/3] 개별 영화 페이지 상세 수집 시작...")
    enriched = enrich_movies(movies)

    # STEP 4
    print("\n[3/3] CSV 저장 중...")
    save(enriched, "rt_certified_fresh_full.csv")


if __name__ == "__main__":
    main()

🍅  Rotten Tomatoes 상세 스크래퍼
    필터: Certified Fresh (비평가) + Verified Hot (관객)

[1/3] 브라우즈 페이지 로드 중...
✅ 31편 발견
   • Crime 101  🍅88%  🍿
   • Good Luck, Have Fun, Don't Die  🍅84%  🍿
   • Redux Redux  🍅98%  🍿
   • The Housemaid  🍅74%  🍿92%
   • 28 Years Later: The Bone Temple  🍅92%  🍿88%
   ... 외 26편

[2/3] 개별 영화 페이지 상세 수집 시작...

📄 개별 상세 페이지 스크래핑 (31편)
   예상 소요 시간: 10~25초

  [  1/31] ✓ Crime 101
  [  2/31] ✓ Good Luck, Have Fun, Don't Die
  [  3/31] ✓ Redux Redux
  [  4/31] ✓ 28 Years Later: The Bone Temple
  [  5/31] ✓ Predator: Badlands
  [  6/31] ✓ The Housemaid
  [  7/31] ✓ Hamnet
  [  8/31] ✓ Eternity
  [  9/31] ✓ Rental Family
  [ 10/31] ✓ One Battle After Another
  [ 11/31] ✓ Sinners
  [ 12/31] ✓ No Other Choice
  [ 13/31] ✓ Sentimental Value
  [ 14/31] ✓ Zootopia 2
  [ 15/31] ✓ Song Sung Blue
  [ 16/31] ✓ Frankenstein
  [ 17/31] ✓ F1 The Movie
  [ 18/31] ✓ Weapons
  [ 19/31] ✓ Wake Up Dead Man: A Knives Out Mystery
  [ 20/31] ✓ Relay
  [ 21/31] ✓ How to Train Your Dragon
  [ 22/31]